# h-DQN experiments on a free GPU (Colab or Kaggle)

Runs `scripts/evaluate.py` for the five h-DQN configs and zips `results/` for download.

**Needs internet** (the harness downloads games and reads official scorecards from the ARC API). Colab has it.
Kaggle accelerated sessions normally have internet *disabled*, so use Kaggle only if its Internet toggle can be switched on.

Before running: zip your repo code on Windows (from the repo root)

`Compress-Archive -Path src,scripts,configs,tests,requirements.txt -DestinationPath ..\RL-Assignment-2026.zip -Force`

and have `RL-Assignment-2026.zip` ready to upload when cell 2 asks.

In [ ]:
# 1. Environment check
import os, sys, glob, subprocess, shutil, time, json
print(sys.version)
assert sys.version_info >= (3, 12), "arc-agi needs Python 3.12+. Pick a runtime/image with Python 3.12 (see the note at the top)."
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-- CPU only (Runtime > Change runtime type > T4 GPU)")
BASE = "/kaggle/working" if os.path.exists("/kaggle") else "/content"

In [ ]:
# 2. Get the repo code (upload RL-Assignment-2026.zip when asked)
ZIP = "RL-Assignment-2026.zip"
cands = glob.glob(f"/kaggle/input/**/{ZIP}", recursive=True) + glob.glob(f"{BASE}/{ZIP}") + glob.glob(ZIP)
if not cands:
    from google.colab import files   # Colab only
    files.upload()
    cands = glob.glob(ZIP) + glob.glob(f"{BASE}/{ZIP}")
assert cands, f"{ZIP} not found"
REPO = f"{BASE}/repo"
shutil.rmtree(REPO, ignore_errors=True)
shutil.unpack_archive(cands[0], REPO)
hit = glob.glob(f"{REPO}/**/scripts/evaluate.py", recursive=True)
assert hit, "scripts/evaluate.py not found inside the zip"
REPO = os.path.dirname(os.path.dirname(hit[0]))
os.chdir(REPO)
print("repo:", REPO, os.listdir(REPO))

In [ ]:
# 3. Install the ARC toolkit (do NOT upgrade numpy/torch here)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arc-agi", "gymnasium"], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
import arc_agi, arcengine
from importlib.metadata import version
print("arc-agi", version("arc-agi"), "| torch", torch.__version__)

In [ ]:
# 4. Smoke test (1 game, 1 seed, 600 actions) -- must finish without a traceback before the long runs
def run_live(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

rc = run_live([sys.executable, "scripts/evaluate.py", "--agent", "hdqn", "--games", "ls20-9607627b",
               "--seeds", "0", "--budget", "600", "--tag", f"smoke_{int(time.time())}", "-v"])
print("exit code", rc)

In [ ]:
# 5. Full runs: 15 dev games x 3 seeds x 5000 actions per config (45 runs each)
CONFIGS = ["hdqn_randmeta", "hdqn_baseline", "hdqn_nov", "hdqn_gru", "hdqn_nov_gru"]
PARALLEL = max(1, min(3, (os.cpu_count() or 2) - 1))   # env stepping is CPU-bound: run configs side by side
SAVE_DIR = None   # e.g. "/content/drive/MyDrive/rl_results" after drive.mount('/content/drive'), so a disconnect loses nothing

def tag_of(n): return json.load(open(f"configs/{n}.json"))["tag"]
for n in CONFIGS:                                       # tags must be unique: clear leftovers of a crashed attempt
    shutil.rmtree(f"results/{tag_of(n)}", ignore_errors=True)

queue, procs, logs = list(CONFIGS), {}, {}
while queue or procs:
    while queue and len(procs) < PARALLEL:
        n = queue.pop(0)
        logs[n] = open(f"/tmp/{n}.log", "w")
        procs[n] = subprocess.Popen([sys.executable, "scripts/evaluate.py", "--config", f"configs/{n}.json"],
                                    stdout=logs[n], stderr=subprocess.STDOUT)
        print("started", n)
    time.sleep(60)
    for n, p in list(procs.items()):
        if p.poll() is not None:
            print("finished", n, "exit code", p.returncode, "(see /tmp/%s.log if non-zero)" % n)
            procs.pop(n)
    prog = {}
    for n in CONFIGS:
        f = f"results/{tag_of(n)}/runs.jsonl"
        prog[n] = sum(1 for _ in open(f)) if os.path.exists(f) else 0
    print(time.strftime("%H:%M:%S"), "runs finished (of 45):", prog)
    if SAVE_DIR:
        shutil.copytree("results", SAVE_DIR, dirs_exist_ok=True)

In [ ]:
# 6. Package everything for download (Colab: Files panel / Kaggle: Output tab)
for n in CONFIGS:
    log = f"/tmp/{n}.log"
    if os.path.exists(log):
        print("=====", n, "-- last lines of its log"); print("".join(open(log).readlines()[-4:]))
out = shutil.make_archive(f"{BASE}/hdqn_results", "zip", "results")
print("wrote", out)
try:
    from google.colab import files
    files.download(out)
except ImportError:
    pass